# Outline

2022/05/22-05/29

- [X] Basic knowledge of trees
    - [x] Representation and storage
    - [X] Common tree types
- [X] Code implementation of common tree operations
- [X] Code implementation of tree traversal algorithms
    - [X] Preorder traversal
    - [X] Inorder traversal
    - [X] Postorder traversal
    - [X] Relationship between the three
- [X] DFS and BFS algorithm implementation
- [X] Targeted practice on iterative methods
- [X] Time/space complexity analysis
- [X] Classic Leetcode examples
- [X] My summary


Reference links:
- [leetcode cookbook tree](https://books.halfrost.com/leetcode/ChapterTwo/Tree/)
- [leetcode cookbook DFS](https://books.halfrost.com/leetcode/ChapterTwo/Depth_First_Search/)
- [leetcode cookbook BFS](https://books.halfrost.com/leetcode/ChapterTwo/Breadth_First_Search/)


# Basic Knowledge of Trees

Structure: parent and child nodes, with a hierarchical relationship
Storage/implementation: linked-list style — each node stores its own value plus its children
Common basic tree types:
- Binary tree: a parent node has at most two children
- Binary Search Tree (BST): binary tree + every value in the left subtree is less than the parent, every value in the right subtree is greater than the parent. Operations:
    - Insert/build: starting from the root, if the value is smaller than the current node, go to the left child; otherwise go to the right child. Create the node once the target position doesn't exist. Ignoring the traversal/search itself and only counting the insertion, complexity is O(1); counting the whole process, it's O(logN) to O(N)
    - Delete: deletion must not break the relative ordering of the left/right children and the parent. Complexity is O(logN) to O(N); ignoring the search and only counting the deletion itself, it's O(1)
        1. Find the node that should take this position after deletion (call it the "replacement node").
        2. Replace this node's value with the replacement node's value.
        3. Since the replacement node's left child is guaranteed to be empty, this substitution can always be completed in this one step, no further looping needed.
        How do you find the replacement node?
        - If the node has no children, the replacement node is None
        - If the node has exactly one child, the replacement node is that child
        - If the node has two children, find the leftmost node of the right subtree.
    - Search: complexity is O(logN) to O(N) (worst case degenerates into a linked list; this is why AVL trees / balanced BSTs were developed)
    - Difference from a heap:
        - By definition: a (min) heap requires any node's value to be less than or equal to its children's values. A BST requires: left child < parent < right child.
        - Insertion: a heap inserts at the end, then sifts up — swapping with a larger parent. A BST starts from the parent, decides whether to descend into the left or right subtree, until it reaches an empty slot, and inserts there.
        - Deletion: a heap moves the last element to the head, then sifts down — swapping with a smaller child. A BST finds the node to delete, and checks whether its left/right subtrees are empty. If neither is empty, it finds the next node in inorder order and swaps with it, then moves/handles that next node's own left/right children.
- More types: balanced BSTs, tries, segment trees, Fenwick trees (binary indexed trees) — see the next note, Tree and Graph (Part 2: Advanced Trees)


In [1]:
# Definition for a binary tree node.
class TreeNode(object):
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def print(self):
        print(self.val)
        if self.left:
            print(str(self.val) + "->left: ")
            self.left.print()
        if self.right:
            print(str(self.val) + "->right: ")
            self.right.print()

In [22]:
# BST Binary Search Tree implementation

class BST(object):
    def __init__(self):
        self.root = TreeNode() # this is the node pointing to the root
        
    def insert(self, val):
        if self.root.left is None:
            self.root.left = TreeNode(val)
            self.root.val = val + 1
        parent_node = self.root.left
        while True:
            if parent_node.val > val:
                if parent_node.left is None:
                    parent_node.left = TreeNode(val)
                    break
                else:
                    parent_node = parent_node.left
            elif parent_node.val == val:
                break
            else:
                if parent_node.right is None:
                    parent_node.right = TreeNode(val)
                    break
                else:
                    parent_node = parent_node.right
                    
    def findNextBigger(self, node):
        val = node.val
        head = node
        child = head.right
        
        while True:
            if child.left:
                head = child
                child = child.left
            elif child.right:
                head = child
                child = child.right
            else:
                break
                
        return head, child  
        
    def delete(self, val):
        parent_node, _ = self.search(val)
        if parent_node.val > val:
            cur_node = parent_node.left
            if cur_node.left is None:
                parent_node.left = cur_node.right
            elif cur_node.right is None:
                parent_node.left = cur_node.left
            else:
                to_replace_parent_node, to_replace_node = self.findNextBigger(cur_node)
                cur_node.val = to_replace_node.val
                if to_replace_parent_node.right == to_replace_node:
                    to_replace_parent_node.right = to_replace_node.right
                else:
                    to_replace_parent_node.left = to_replace_node.right
                
        else:
            cur_node = parent_node.right
            if cur_node.left is None:
                parent_node.right = cur_node.right
            elif cur_node.right is None:
                parent_node.right = cur_node.left
            else:
                to_replace_parent_node, to_replace_node = self.findNextBigger(cur_node)
                cur_node.val = to_replace_node.val
                if to_replace_parent_node.right == to_replace_node:
                    to_replace_parent_node.right = to_replace_node.right
                else:
                    to_replace_parent_node.left = to_replace_node.right
        
    def search(self, val):
        # return two values, one is the parent node, the other is whether to find
        if self.root.left is None:
            return self.root, False
        parent_node = self.root
        cur_node = parent_node.left
        while True:
            if cur_node.val > val:
                if cur_node.left is None:
                    return cur_node, False
                else:
                    parent_node = cur_node
                    cur_node = cur_node.left
            elif cur_node.val == val:
                return parent_node, True
            else:
                if cur_node.right is None:
                    return cur_node, False
                else:
                    parent_node = cur_node
                    cur_node = cur_node.right
    def print(self):
        self.root.left.print()

In [23]:
tree = BST()
nums = [6,4,2,5,8,7,10]
for num in nums:
    tree.insert(num)
tree.print()
tree.delete(6)
print('After deleting node')
tree.print()

6
6->left: 
4
4->left: 
2
4->right: 
5
6->right: 
8
8->left: 
7
8->right: 
10
After deleting node
7
7->left: 
4
4->left: 
2
4->right: 
5
7->right: 
8
8->right: 
10


## Tree Traversal

There are four common traversal orders, each with both a recursive and an iterative implementation
- Recursive: repeated calls
- Iterative: traversal, usually implemented with the help of a stack
They are:
- Preorder: root, left, right
    - [144Binary Tree Preorder Traversal](https://leetcode.cn/problems/binary-tree-preorder-traversal/)
    - Iterative: what happens at each step? Pop the current element, store its value, then push its right child and then its left child onto the stack
- Inorder: left, root, right
    - [94Binary Tree Inorder Traversal](https://leetcode.cn/problems/binary-tree-inorder-traversal/)
    - 🌟Iterative: what happens at each step? First handle the elements that still need to be pushed: push the element, then push all of its left descendants. Next, pop the top of the stack and store its value. Set its right child as the next element to push. Same as preorder: an element being popped is when its value gets stored, meaning it and its whole left subtree have been visited. Different from preorder: the left subtree gets pushed first. Two nested loops.
    - Iterative version 2: use a flag to mark whether the left subtree has already been pushed, in order to tell apart "store the value" from "push the left subtree."
- Postorder: left, right, root
    - [145Binary Tree Postorder Traversal](https://leetcode.cn/problems/binary-tree-postorder-traversal/)
    - [590n-ary tree postorder traversal](https://leetcode.cn/problems/n-ary-tree-postorder-traversal/)
- Level-order traversal:
    - [X] How is this different from BFS? Level-order traversal IS BFS!
    - [102Binary tree level order traversal](https://leetcode.cn/problems/binary-tree-level-order-traversal/)
        - Approach 1: visit one node at a time. Use a queue to hold the nodes still waiting to be visited. For each node, use a state value to record which level it's on. Whenever the traversal moves down to the next level, append a new list to the result res.
        - Approach 2: visit one level at a time. Each level's result is its own list

Relationship between the preorder, inorder, and postorder traversals of a binary tree:
Any two of them let you derive the third; any two of them let you reconstruct a binary tree. The key to reconstruction is: find the root node, and the traversal results for the left and right subtrees, then construct recursively.

- Reconstructing a binary tree from preorder + inorder (unique): the first element of preorder is the root. Find the root's position in inorder — everything before it in inorder is the left subtree's traversal, everything after it is the right subtree's traversal. Use the length of the left subtree (from inorder) to slice out the left subtree's preorder segment and recursively construct the left subtree; do the same for the right subtree.
    - Reference problem: [(medium)105Construct Binary Tree From Preorder And Inorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-inorder-traversal/)
- Reconstructing a binary tree from preorder + postorder (may not be unique): the first element of preorder is the root, and it's also the last element of postorder;
    - Assuming preorder has a left subtree: the second element of preorder is the root of the left subtree — find its position in postorder, which lets you split off the left subtree. Then recurse on the left/right subtrees.
    - Assuming preorder has no left subtree: then the second element of preorder is the root of the right subtree, and it should appear as the second-to-last element of postorder. If it doesn't appear there, that proves preorder does have a left subtree after all. So, for a parent that has only a single leaf child, the result may not be unique — there's no way to tell whether that child should be the left or the right leaf.
    - Reference problem: [(medium)889Construct Binary Tree From Preorder And Postorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-postorder-traversal/)
- Reconstructing a binary tree from inorder + postorder (unique): the last node of postorder is the leaf node — find the root's position in inorder, everything before the root in inorder is the left subtree's traversal, everything after is the right subtree's traversal. Recurse.
    - Reference problem: [(medium)106Construct Binary Tree From Inorder And Postorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-inorder-and-postorder-traversal/)



In [ ]:
void traverse(TreeNode root) {
    // preorder traversal code
    traverse(root.left);
    // inorder traversal code
    traverse(root.right);
    // postorder traversal code
}

In [ ]:
# preorder recursion version
class Solution:
    def preorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        res = [root.val] # visit the parent node first
        res.extend(self.preorderTraversal(root.left)) # then visit the left child
        res.extend(self.preorderTraversal(root.right)) # finally visit the right child
        return res

# preorder iteration version
class Solution:
    def preorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        stack = [root] # simulates the recursion stack; holds the next element to visit
        res = []
        while stack:
            top = stack.pop()
            res.append(top.val) # visit the parent node first
            if top.right:
                stack.append(top.right) # push the right child first, so it's popped later
            if top.left:
                stack.append(top.left) # push the left child last, so it's popped first
        return res

# n-ary preorder recursion version
class Solution:
    def preorder(self, root: 'Node') -> List[int]:
        if root is None:
            return []
        res = []
        res.append(root.val)
        for child in root.children:
            res.extend(self.preorder(child))
        return res

In [ ]:
# inorder recursion version
class Solution:
    def inorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        res = []
        res.extend(self.inorderTraversal(root.left))
        res.append(root.val)
        res.extend(self.inorderTraversal(root.right))
        return res

# inorder iteration version
class Solution:
    def inorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        stack = []
        to_enter = root
        res = []
        while stack or to_enter:
            while to_enter:
                stack.append(to_enter)
                to_enter = to_enter.left
            head = stack.pop()
            res.append(head.val)
            to_enter = head.right
        return res

# inorder iteration version 2
class Solution:
    def inorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        stack = [(root, 0)] # use a flag to mark whether the left subtree has already been pushed
        res = []
        while stack:
            head = stack.pop()
            if head[1] == 0: # push the left subtree
                to_enter = head[0].left
                stack.append((head[0], 1))
                while to_enter:
                    stack.append((to_enter, 0))
                    to_enter = to_enter.left
            else: # store the value
                res.append(head[0].val)
                if head[0].right:
                    stack.append((head[0].right, 0))
        return res

In [ ]:
# post order recursion version
class Solution:
    def postorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        res = []
        if not root:
            return []
        res.extend(self.postorderTraversal(root.left))
        res.extend(self.postorderTraversal(root.right))
        res.append(root.val)
        return res

# postorder iteration version
class Solution:
    def postorderTraversal(self, root: Optional[TreeNode]) -> List[int]:
        res = []
        stack = []
        stack.append((root, 0)) # flag-based approach
        while stack:
            top, cmd = stack.pop()
            if top is None:
                continue
            if cmd == 0:
                stack.append((top, 1))
                stack.append((top.right, 0))
                stack.append((top.left, 0))
            else:
                res.append(top.val)
        return res

# n-ary postorder
class Solution:
    def postorder(self, root: 'Node') -> List[int]:
        if root is None:
            return []
        res = []
        for child in root.children:
            res.extend(self.postorder(child))
        res.append(root.val)
        return res

In [ ]:
# 102 level order traversal（BFS）Approach 1: visit one node at a time
from collections import deque
class Solution:
    def levelOrder(self, root: TreeNode) -> List[List[int]]:
        if not root:
            return []
        cur_layer = -1
        q = deque([[root, 0]]) # use a queue to hold nodes waiting to be visited. Track a state: which layer/level it's on
        res = []
        while q:
            top = q.popleft()
            if top[1] != cur_layer: # when moving to the next layer, append a new list to res
                res.append([top[0].val])
                cur_layer = top[1]
            else:
                res[-1].append(top[0].val)
            if top[0].left:
                q.append([top[0].left, cur_layer + 1])
            if top[0].right:
                q.append([top[0].right, cur_layer + 1])
        return res


# Approach 2 BFS: visit one layer at a time
class Solution:
    def levelOrder(self, root: TreeNode) -> List[List[int]]:
        if not root:
            return []
        res = []
        next_layer = [root]
        while next_layer:
            res.append([])
            cur_layer = next_layer
            next_layer = []
            for node in cur_layer: # traverse the current layer, and collect the nodes for the next layer
                res[-1].append(node.val)
                if node.left:
                    next_layer.append(node.left)
                if node.right:
                    next_layer.append(node.right)
        return res

## Tree Search

BFS and DFS are traversals of **non-linear structures**!!!

Reference:
https://developer.aliyun.com/article/756316

Iterative and recursive implementations
- DFS traversal uses recursion
- BFS traversal uses a queue data structure. General pattern:
    - Push the root node into the queue first.
    - Queue: holds the nodes still waiting to be visited
    - Each time, pop one element from the queue and use it to update the result.
    - Decide whether to push its left/right children into the queue



Depth-first traversal, summed up in one sentence

> As long as there is a path forward, keep going forward; only turn back once there's no path left

There are two situations that count as "no path left":

- Hitting a wall;
- Hitting a path already walked;


Applications of BFS
- Level-order traversal
- Shortest path

Applications of DFS
- Whether a feasible solution exists, e.g. maze solving


In [ ]:
# BFS Implementation
class
from collections import deque
def BFS(root):
    if not root:
        return []
    res = []
    to_search = deque([root])
    while to_search:
        head = to_search.popleft()
        res.append(head.val)
        if head.left:
            to_search.append(head.left)
        if head.right:
            to_search.append(head.right)
    return res

# pseudocode, general pattern
# def BFS(root):
#     if isnull(root):
#         return
#     res = set_default_result(res)
#     push_node_in_queue(root, queue)
#     while is_not_empty(queue):
#         head = get_head(queue)
#         update_result(head, res)
#         if head.left:
#             push_node_in_queue(head.left, queue)
#         if head.right:
#             push_node_in_queue(head.right, queue)
#     return res

In [ ]:
# DFS Recursion Implementation
def dfs(root):
    if not root:
        return []
    res = []
    res.extend(self.dfs(root.left))
    res.extend(self.dfs(root.right))
    res.append(root.val)
    return res

# iteration implementation
def dfs(root):
    if not root:
        return
    stack = [root]
    while stack:
        head = stack.pop()
        process(head)
        if head.right: # push the right node first
            stack.append(head.right)
        if head.left: # then push the left node
            stack.append(head.left)


# pseudocode, general recursive pattern
# def dfs(root):
#     if isnull(root):
#         return
#     process(root)
#     dfs(root.left)
#     dfs(root.right)

# Leetcode Examples

## Tree Traversal


- [(easy)897increasing order search tree](https://leetcode.cn/problems/increasing-order-search-tree/). Approach: a variant of preorder traversal. The difference: use a tree node that only has a right child (instead of an array) to store the traversal result.
- [(medium)105Construct Binary Tree From Preorder And Inorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-inorder-traversal/)
    - Approach 1: recursive construction — each time, find the current root's position in inorder, then recursively construct the left and right subtrees. Worst-case time complexity is O(n^2), when there's only a left subtree. The costly part is: finding the current root's position.
    - Approach 2: record the mapping from preorder node to its position in inorder, then construct recursively. Time complexity is O(n)
    - [X] 💡Approach 3: iterative method: [reference solution](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-inorder-traversal/solution/cong-qian-xu-yu-zhong-xu-bian-li-xu-lie-gou-zao-9/)
- [(medium)106Construct Binary Tree From Inorder And Postorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-inorder-and-postorder-traversal/)
    - Approach 1: recursive construction, using a hashmap to reduce complexity.
    - [X] 💡Approach 2: iterative method, similar to the iterative method for #105.
- [(medium)889Construct Binary Tree From Preorder And Postorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-postorder-traversal/).
    - Approach 1: recursive construction + hashmap. The key is: using the two arrays to determine the root, the subarray corresponding to the left subtree, and the subarray corresponding to the right subtree. How? In preorder, the root of the left subtree is the second element of the preorder array; find that number's position in postorder, and that tells you how many nodes the left subtree has. Then slice out the corresponding segments for the left/right subtrees from preorder and postorder, and solve recursively.
    - [X] 💡Approach 2: iterative method, [see the reference solution's comments](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-postorder-traversal/solution/gen-ju-qian-xu-he-hou-xu-bian-li-gou-zao-er-cha-sh/)



In [ ]:
# 105 Approach 1: recursive construction. Worst-case time complexity is O(n^2)
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> TreeNode:
        if not preorder:
            return None
        left_cnt = 0
        for node in inorder: # this step is relatively slow
            if node == preorder[0]:
                break
            left_cnt += 1
        root = TreeNode(preorder[0])
        root.left = self.buildTree(preorder[1:1+left_cnt], inorder[:left_cnt])
        root.right = self.buildTree(preorder[1+left_cnt:], inorder[1+left_cnt:])
        return root


# Approach 2: first build a mapping from preorder value to its inorder index, time complexity is O(n)
# Time: 120ms, Space: 88MB
class Solution:
    def recurseBuild(self, preorder, inorder, pre2in, diff):
        if not preorder:
            return None
        root = TreeNode(preorder[0])
        left_cnt = pre2in[preorder[0]] - diff
        root.left = self.recurseBuild(preorder[1:1+left_cnt], inorder[:left_cnt], pre2in, diff)
        root.right = self.recurseBuild(preorder[1+left_cnt:], inorder[1+left_cnt:], pre2in, 1 + left_cnt + diff)
        return root

    def buildTree(self, preorder: List[int], inorder: List[int]) -> TreeNode:
        pre2in = {}
        for i, node in enumerate(inorder):
            pre2in[node] = i
        return self.recurseBuild(preorder, inorder, pre2in, 0)

# Approach 3: iterative: to be learned

In [ ]:
# 106 Approach 1: recursive call
class Solution:
    def recurseBuild(self, inorder, postorder, post2in, diff):
        if not inorder:
            return None
        left_cnt = post2in[postorder[-1]] - diff
        root = TreeNode(postorder[-1])
        root.left = self.recurseBuild(inorder[:left_cnt], postorder[:left_cnt], post2in, diff)
        root.right = self.recurseBuild(inorder[left_cnt+1: ], postorder[left_cnt: -1], post2in, diff + left_cnt + 1)
        return root

    def buildTree(self, inorder: List[int], postorder: List[int]) -> TreeNode:
        post2in = {}
        for i, node in enumerate(inorder):
            post2in[node] = i
        return self.recurseBuild(inorder, postorder, post2in, 0)
# Approach 2: iterative: to be learned

In [ ]:
# 889 Approach 1: recursive construction
class Solution:
    def recurseBuild(self, preorder, postorder, diff):
        if not preorder:
            return None
        root = TreeNode(preorder[0])
        if len(preorder) == 1:
            return root
        left_cnt = self.pre2post[preorder[1]] - diff + 1 # the root of the left subtree is the second element of preorder; find its position in postorder
        # slice out the segments corresponding to the left/right subtrees from preorder and postorder, and solve recursively
        root.left = self.recurseBuild(preorder[1:1+left_cnt], postorder[:left_cnt], diff)
        root.right = self.recurseBuild(preorder[1+left_cnt:], postorder[left_cnt:-1], diff + left_cnt)
        return root

    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> TreeNode:
        self.pre2post = {}
        for i, node in enumerate(postorder):
            self.pre2post[node] = i
        return self.recurseBuild(preorder, postorder, 0)

# Approach 2: iterative solution, to be learned
class Solution {
public:
    # The essential difference in visiting order between preorder and postorder traversal
    # Preorder: take a step, record a step. Postorder: walk all the way to the end each time, then step back, then record.
    # So we use preorder directly to build the tree, and use postorder to "return" — equivalent to a preorder recursive visit, using postorder traversal to return to the previous node
    # Postorder traversal acts as the "return stack" within the preorder traversal
    TreeNode* constructFromPrePost(vector<int>& pre, vector<int>& post) {
        vector<TreeNode*> stack;
        stack.push_back(new TreeNode(pre[0]));
        for(int i = 1, j = 0; i < pre.size(); ++i){
            TreeNode* node = new TreeNode(pre[i]);
            while(stack.back()->val == post[j])
                stack.pop_back(),j++;
            if(stack.back()->left == nullptr) stack.back()->left = node;
            else stack.back()->right = node;
            stack.push_back(node);
        }
        return stack[0];
    }
};

## Tree Search

[reference solution](https://leetcode.cn/problems/binary-tree-level-order-traversal/solution/bfs-de-shi-yong-chang-jing-zong-jie-ceng-xu-bian-l/)
[wechat article](https://mp.weixin.qq.com/s?__biz=MzA5ODk3ODA4OQ==&mid=2648167208&idx=1&sn=d8118c7c0e0f57ea2bdd8aa4d6ac7ab7&chksm=88aa236ebfddaa78a6183cf6dcf88f82c5ff5efb7f5c55d6844d9104b307862869eb9032bd1f&token=1064083695&lang=zh_CN#rd)

### BFS
- [(easy)637Average of Levels in Binary Tree](https://leetcode.cn/problems/average-of-levels-in-binary-tree/)
    - Approach: BFS. The queue holds all the nodes of a single level, and the operations are performed level-by-level rather than node-by-node.
- [(medium)103Binary Tree Zigzag Level Order Traversal](https://leetcode.cn/problems/binary-tree-zigzag-level-order-traversal/). Approach: standard BFS; the only difference is reversing each level's stored result based on the direction indicator.
- [(medium)199Binary Tree Right Side View](https://leetcode.cn/problems/binary-tree-right-side-view/). Approach: BFS. All nodes on each level must be stored for traversing the next level, because it's possible that the rightmost node of a level has no children — meaning any node on that level could turn out to be the parent of the next level's rightmost node. When keeping the result for a level, just keep the rightmost value.
- [(medium)515Find Largest Value in Each Tree Row](https://leetcode.cn/problems/find-largest-value-in-each-tree-row/). BFS: when saving each level's result, take the max value.
- [(medium)54201 Matrix](https://leetcode.cn/problems/01-matrix/)
    - Approach 1: run BFS starting from every point to find the minimum distance. Times out. Time complexity is O(m^2 n^2).
    - Approach 2: first find all the "root" nodes (the cells whose matrix value is 0), then run BFS. While finding neighbors, update the distance as soon as a neighbor is found, to avoid redundant searching.

- [(medium)994Rotting Oranges](https://leetcode.cn/problems/rotting-oranges/)
    - Approach: BFS, with an extra check needed for whether all the fresh oranges ended up rotten.
- [(medium)1162As Far From Land As Possible](https://leetcode.cn/problems/as-far-from-land-as-possible/)
    - Approach: BFS — the maximum distance from land to the nearest water. If you start from each water cell, every water cell can find such a distance; iterate over all water cells and take the max. That approach has fairly high complexity. Flip the idea: starting from every land cell simultaneously is equivalent to asking for the minimum distance needed for land to reach every water cell.

- [(hard)126Word Ladder II](https://leetcode.cn/problems/word-ladder-ii/)
    - Approach 1: overall idea is to build a tree, then BFS: find the shortest distance from the root node (beginWord) to the end node (endWord). A node's children are all the not-yet-used words that differ from it by exactly one letter. Search recursively until endWord is reached.
    - Approach 2: bidirectional BFS — search downward from beginWord and upward from endWord simultaneously, and stop once they overlap.

In [2]:
# 637 BFS
from collections import deque
class Solution:
    def averageOfLevels(self, root: Optional[TreeNode]) -> List[float]:
        q = deque([[root]])
        res = []
        while q:
            nodes = q.popleft()
            cur_layer_sum = 0
            next_layer_nodes = []
            for node in nodes:
                cur_layer_sum += node.val
                if node.left:
                    next_layer_nodes.append(node.left)
                if node.right:
                    next_layer_nodes.append(node.right)
            res.append(cur_layer_sum / 1.0 / len(nodes))
            if next_layer_nodes:
                q.append(next_layer_nodes)
        return res

NameError: name 'Optional' is not defined

In [ ]:
# 103 zigzag
from collections import deque
class Solution:
    def zigzagLevelOrder(self, root: TreeNode) -> List[List[int]]:
        left2right = 1
        if not root:
            return []
        q = deque([[root]])
        res = []
        while q:
            cur_layer = q.popleft()
            next_layer = []
            cur_layer_res = []
            for node in cur_layer:
                cur_layer_res.append(node.val)
                if node.left:
                    next_layer.append(node.left)
                if node.right:
                    next_layer.append(node.right)
            # difference from standard BFS: only in how this level's result is stored, based on the direction indicator
            if left2right == 1:
                res.append(cur_layer_res)
            else:
                res.append(cur_layer_res[::-1])
            left2right = 1 - left2right
            if next_layer:
                q.append(next_layer)
        return res

In [ ]:
# 199
class Solution:
    def rightSideView(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        res = []
        next_layer = [root]
        while next_layer:
            cur_layer = next_layer
            cur_layer_val = []
            next_layer = []
            for node in cur_layer:
                cur_layer_val.append(node.val)
                if node.left:
                    next_layer.append(node.left)
                if node.right:
                    next_layer.append(node.right)
            # difference from standard BFS: when keeping this level's result, just keep the rightmost value.
            res.append(cur_layer_val[-1])
        return res

In [ ]:
# 515
class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        next_layer = [root]
        res = []
        while next_layer:
            cur_layer = next_layer
            next_layer = []
            cur_layer_res = []
            for node in cur_layer:
                cur_layer_res.append(node.val)
                if node.left:
                    next_layer.append(node.left)
                if node.right:
                    next_layer.append(node.right)
            # difference from standard BFS: when saving each level's result, take the max value.
            res.append(max(cur_layer_res))
        return res

In [ ]:
# 542 run BFS from every point. Times out
class Solution:
    def GetNeighsCheckZero(self, i, j, m, n, mat):
        neighs = []
        for next_i, next_j in [[i - 1,j], [i + 1, j], [i, j - 1], [i, j+1]]:
            if next_i < 0 or next_i >= m or next_j < 0 or next_j >= n:
                continue
            if mat[next_i][next_j] == 0:
                return [], True
            neighs.append([next_i, next_j])
        return neighs, False

    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m = len(mat)
        n = len(mat[0])
        res = []
        for i in range(m):
            layer_res = []
            for j in range(n):
                if mat[i][j] == 0:
                    layer_res.append(0)
                    continue
                node_dist = 0
                next_nodes = [[i, j]]
                while next_nodes:
                    node_dist += 1
                    cur_nodes = next_nodes
                    next_nodes = []
                    for node in cur_nodes:
                        neighbors, got_zero = self.GetNeighsCheckZero(node[0], node[1], m, n, mat)
                        if got_zero:
                            next_nodes = []
                            break
                        next_nodes.extend(neighbors)
                layer_res.append(node_dist)
            res.append(layer_res)
        return res

# Approach 2: first find all the root nodes, then BFS
class Solution:
    def initialize(self, mat, m, n):
        # find all the root nodes: the cells whose matrix value is 0
        dist = []
        roots = []
        for i in range(m):
            dist.append([])
            for j in range(n):
                if mat[i][j] == 0:
                    roots.append([i,j])
                    dist[-1].append(0)
                else:
                    dist[-1].append(-1)
        return dist, roots

    def findNeighs(self, node, dist, cur_dist, m, n):
        i, j = node[0], node[1]
        neighs = []
        for next_i, next_j in [[i-1, j], [i + 1, j], [i, j - 1], [i, j + 1]]:
            if next_i < 0 or next_i >= m or next_j < 0 or next_j >= n:
                continue
            if dist[next_i][next_j] < 0:
                neighs.append([next_i, next_j])
                dist[next_i][next_j] = cur_dist # update the dist matrix as soon as a neighbor is found, to avoid redundant work
        return neighs


    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m = len(mat)
        n = len(mat[0])
        dist, roots = self.initialize(mat, m, n)
        next_layer = roots
        cur_dist = 0
        while next_layer:
            cur_dist += 1
            cur_layer = next_layer
            next_layer = []
            for node in cur_layer:
                neighs = self.findNeighs(node, dist, cur_dist, m , n)
                next_layer.extend(neighs)
        return dist

In [ ]:
# 994
class Solution:
    def initalize(self, grid, m, n):
        cnt = 0
        roots = []
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    cnt += 1
                if grid[i][j] == 2:
                    roots.append([i, j])
        return cnt, roots

    def rotten(self, node, grid, m, n):
        neighs = []
        for next_i, next_j in [[node[0] -1 , node[1]], [node[0]+1, node[1]], [node[0], node[1] - 1], [node[0], node[1] + 1]]:
            if next_i < 0 or next_i >= m or next_j < 0 or next_j>= n or grid[next_i][next_j] != 1:
                continue
            grid[next_i][next_j] = 2
            neighs.append([next_i, next_j])
        return neighs

    def orangesRotting(self, grid: List[List[int]]) -> int:
        m = len(grid)
        n = len(grid[0])
        fresh_cnt, next_layer = self.initalize(grid, m, n)
        minutes = -1
        rotten_cnt = 0 # used together with fresh_cnt, to check whether all the fresh oranges ended up rotten
        while next_layer:
            cur_layer = next_layer
            minutes += 1 # update the time
            next_layer = []
            for orange in cur_layer:
                new_rottens = self.rotten(orange, grid, m, n)
                next_layer.extend(new_rottens)
                rotten_cnt += len(new_rottens)
        if fresh_cnt == rotten_cnt:
            return max(minutes, 0)
        return -1

In [ ]:
# 1162
class Solution:
    def findAllLand(self, grid, n):
        lands = []
        for i in range(n):
            for j in range(n):
                if grid[i][j] == 1:
                    lands.append([i, j])
        return lands

    def findNeighs(self, land, grid, n):
        i, j = land[0], land[1]
        neighs = []
        for next_i, next_j in [[i-1,j], [i+1,j], [i, j-1], [i, j+1]]:
            if next_i < 0 or next_i >= n or next_j < 0 or next_j >= n or grid[next_i][next_j] == 1:
                continue
            neighs.append([next_i, next_j])
            grid[next_i][next_j] = 1 # update this water cell to land
        return neighs

    def maxDistance(self, grid: List[List[int]]) -> int:
        n = len(grid)
        next_layer = self.findAllLand(grid, n)
        if len(next_layer) == n*n:
            return -1
        dist = -1
        while next_layer:
            cur_layer = next_layer
            next_layer = []
            dist += 1 # update the distance
            for land in cur_layer:
                water_neighs = self.findNeighs(land, grid, n)
                next_layer.extend(water_neighs)
        return dist

In [ ]:
# 126 Approach 1: unidirectional BFS
from collections import deque
class Solution:
    def __init__(self):
        self.node2children = {}
        self.visited = {}
        self.end = None

    def OneDiff(self, word1, word2):
        diff_cnt = 0
        for i, c in enumerate(word1):
            if c != word2[i]:
                diff_cnt += 1
        return diff_cnt == 1

    def BuildChildren(self, beginWord, wordList, endWord):
        self.node2children[beginWord] = []
        self.visited[beginWord] = False
        for word in wordList:
            if self.OneDiff(beginWord, word):
                self.node2children[beginWord].append(word)
        for word in wordList:
            if word == endWord:
                continue
            self.node2children[word] = []
            self.visited[word] = False
            for other_word in wordList:
                if self.OneDiff(word, other_word):
                    self.node2children[word].append(other_word)

    def BFS(self, root, former_path):
        next_layer = [[root, former_path]]
        found_end = False
        while True: # breadth-first search, level by level
            cur_layer = next_layer
            next_layer = []
            for node, former_path in cur_layer:
                # print(node, former_path)
                self.visited[node] = True
                for child in self.node2children[node]:
                    if child == self.end:
                        found_end = True
                    elif self.visited[child]: # skip nodes already visited
                        continue
                    next_layer.append([child, former_path + [child]])
            if found_end: # exit as soon as endWord is first reached — this is the shortest path
                return [path[1] for path in next_layer if path[1][-1] == self.end]
            if not next_layer:
                break
        return []

    def findLadders(self, beginWord: str, endWord: str, wordList: List[str]) -> List[List[str]]:
        if endWord not in wordList:
            return []
        self.end = endWord
        self.BuildChildren(beginWord, wordList, endWord) # build the parent-child relationships
        return self.BFS(beginWord, [beginWord]) # breadth-first search

In [ ]:
# 126 Approach 2: bidirectional BFS
from collections import deque
class Solution:
    def __init__(self):
        self.node2children = {}
        self.begin_visited = {}
        self.end_visited = {}

    def OneDiff(self, word1, word2):
        diff_cnt = 0
        for i, c in enumerate(word1):
            if c != word2[i]:
                diff_cnt += 1
        return diff_cnt == 1

    def BuildChildren(self, wordList):
        for word in wordList:
            self.node2children[word] = []
            self.begin_visited[word] = False
            self.end_visited[word] = False
            for other_word in wordList:
                if self.OneDiff(word, other_word):
                    self.node2children[word].append(other_word)

    def mergePaths(self, meet_nodes, former_paths, end_paths):
        res = []
        for meet_node in meet_nodes:
            former_has = []
            later_has = []
            for p in former_paths:
                if p[-1] == meet_node:
                    former_has.append(p)
            for p in end_paths:
                if p[0] == meet_node:
                    later_has.append(p[1:])
            for fp in former_has:
                for lp in later_has:
                    res.append(fp+lp)
        return res


    def BFS(self, begin_path, end_path):
        begin_next_layer = [begin_path]
        end_next_layer = [end_path]
        meet_nodes = []
        while True:
            begin_cur_layer = begin_next_layer
            end_cur_layer = end_next_layer
            begin_next_layer, end_next_layer = [], []
            cur_visit = set() # track which nodes were visited in this round
            for former_path in begin_cur_layer:
                node = former_path[-1]
                if self.end_visited[node]:
                    meet_nodes.append(node)
                else:
                    for child in self.node2children[node]:
                        if self.begin_visited[child]:
                            continue
                        cur_visit.add(child)
                        begin_next_layer.append(former_path + [child])
            for child in cur_visit:
                self.begin_visited[child] = True # update the visited status for these nodes
            if meet_nodes:
                return self.mergePaths(set(meet_nodes), begin_cur_layer, end_cur_layer)
            cur_visit = set()
            for former_path in end_cur_layer:
                node = former_path[0]
                if self.begin_visited[node]:
                    meet_nodes.append(node)
                else:
                    for child in self.node2children[node]:
                        if self.end_visited[child]:
                            continue
                        cur_visit.add(child)
                        end_next_layer.append([child] + former_path)
            for child in cur_visit:
                self.end_visited[child] = True

            if meet_nodes:
                return self.mergePaths(set(meet_nodes), begin_next_layer, end_cur_layer)
            if not begin_next_layer and not end_next_layer:
                return []

    def findLadders(self, beginWord: str, endWord: str, wordList: List[str]) -> List[List[str]]:
        if endWord not in wordList:
            return []
        all_words = wordList + [beginWord]
        if beginWord in wordList:
            all_words = wordList
        self.BuildChildren(all_words)
        self.begin_visited[beginWord] = True
        self.end_visited[endWord] = True
        return self.BFS([beginWord], [endWord])

## DFS

- [(easy)543Diameter of Binary Tree](https://leetcode.cn/problems/diameter-of-binary-tree/)
    - Approach: first, what's special about a "path"? A path always has some root node, and its length is: the distance from that root to the deepest left leaf, plus the distance from that root to the deepest right leaf. That root doesn't have to be the true root of the whole tree — it can be any node in the tree. So, for any node in the tree, we need two pieces of information: the max depth of its left branch, and the max depth of its right branch. Sum them to update a global diameter maximum, and take their max to return to the parent. This is clearly a DFS, where the return value is the node's max depth, and a global variable is maintained and updated during the recursion.

- [(medium)669Trim a Binary Search Tree](https://leetcode.cn/problems/trim-a-binary-search-tree/). First think clearly about what operation to perform on the current node: we should trim its left and right subtrees, not trim the node itself — because trimming the node itself would require modifying the branch pointing to it from its parent. Trimming the node itself should be left to its parent to handle. To make that possible, create a dummy root node and make the real root its left child. How do you trim a child node?
    - When a child's value is less than low, replace it with that child's right child;
    - When a child's value is greater than high, replace it with that child's left child;
  Note the recursion's base case: the node is empty.

- 🌟[(medium)99Recover Binary Search Tree](https://leetcode.cn/problems/recover-binary-search-tree/)
    - Approach: DFS + inorder traversal

- [(medium)1372Longest Zigzag Path in a Binary Tree](https://leetcode.cn/problems/longest-zigzag-path-in-a-binary-tree/)
    - Approach: DFS.
        - What operation does each node perform? Take the values from its two children and use them to update its own value. There are two values to update:
            - The longest zigzag path going left from this node, using the left child's rightward zigzag length + 1;
            - The longest zigzag path going right from this node, using the right child's leftward zigzag length + 1.
        - At the same time, update a global variable during the DFS
        - What's the base case for the recursion? The node is empty. You can add an early-stop check here to save some time.
- [(medium)1367Linked List in Binary Tree](https://leetcode.cn/problems/linked-list-in-binary-tree/)
    - Approach 1: for every node in the tree, try two kinds of DFS expansions: assume it is the node matching `head` (starting fresh from the beginning of the list) / or continue the existing match (continuing from wherever we'd got to in the list, moving to the next node that should match). Here I try both cases separately for the left child and the right child. Time complexity is quite high.
    - Approach 2: the overall idea is still to try both kinds of DFS expansion, but with a different order, plus early pruning. Separate the two kinds of DFS:
        - One is: DFS starting fresh from the original head. The "restart" attempt (realigning the current node with the original head) happens here.
        - The other is: check whether the subtree starting at the current root is currently matching the list starting from head. If the current root's value differs from head's value, stop early. This kind of DFS never does any "restart" (realigning with the original head).
      And in terms of order, the second kind runs first, then the first kind.
- [(hard)297Serialize and Deserialize Binary Tree](https://leetcode-cn.com/problems/serialize-and-deserialize-binary-tree/). The main idea is to let the deserialization process guide how serialization is done.
    - Approach 1: use preorder traversal for serialization, and DFS + bracket matching for deserialization. Bracket matching is used to locate the substring for the left subtree and the substring for the right subtree, letting us construct recursively. Average-case time complexity is O(n log n): at each step, determining the left/right substrings requires scanning, on average, about half the length of the remaining serialized string, and this happens about log n times on average.
    - Approach 2: deserialize with a single direct pass, using a stack to hold the nodes that still need a child constructed. This brings the time complexity down to O(n). Details:
        - Each object stored on the stack is [node, state], where node is the node that currently needs its left or right child constructed; state 0 means the left child still needs to be constructed, state 1 means the right child still needs to be constructed.
        - For each element encountered during the traversal:
            - If it is null, pop the top-of-stack element, and based on the state stored on it, attach it as the left or right child of the (new) top-of-stack element.
                - If it is attached as a left child, push it back onto the stack and change its state to 1.
                - If it is attached as a right child, treat the parent node as the newly-completed child node, continue popping the top of the stack, and repeat the child-attaching operation.
            - If it is not null, create a new node, push it onto the stack, and set its state to 0.
- [(hard)332Reconstruct Itinerary](https://leetcode.cn/problems/reconstruct-itinerary/)
    - Approach 1: DFS + greedy. From the current airport, among all possible next stops, pick the alphabetically smallest one and DFS into it. If a full itinerary is found this way, it must be the globally alphabetically-smallest one, so we can return immediately. The tricky part: how do you find all possible next-stop airports? Here I maintain two dictionaries: one for which airports can still be flown to, and one for which have already been used — the difference gives the remaining reachable airports.
    - [ ] Approach 2: Eulerian path, to be learned — see the note Tree and Graph (Part 3: Graphs)


- [(hard)987Vertical Order Traversal of a Binary Tree](https://leetcode.cn/problems/vertical-order-traversal-of-a-binary-tree/)
    - Approach: use BFS to collect each node's col, row, and val, then sort by col first, then row, then val. Save the sorted result as required. Time complexity: O(n log n), where BFS is O(n), sorting is O(n log n), and saving the result is O(n); space complexity O(n). There are many variations on this approach: e.g. using DFS to collect the info, using a heap to sort, etc.
- [(medium)Interview Question 04.06 Successor](https://leetcode.cn/problems/successor-lcci/). Find the first node in a BST whose value is greater than the target value — i.e., do a search on the BST. Compare against the current node, then decide whether to go into the left or right subtree, keeping a global variable to track the smallest node found so far that's greater than the target.

- [(medium)652 Find Duplicate Subtrees](https://leetcode.cn/problems/find-duplicate-subtrees/)
    - Approach: DFS in postorder position + serialization. Serialize each subtree to describe it as a string — otherwise, if you just used the Node objects directly, you couldn't detect duplicates, since trees with identical content would still be judged as different.

In [ ]:
# 543
class Solution:
    def __init__(self):
        self.max = -1
    def dfs(self, node):
        if not node:
            return -1
        left_depth = self.dfs(node.left)
        right_depth = self.dfs(node.right)
        self.max = max(self.max, left_depth + right_depth + 2)
        return max(left_depth, right_depth) + 1

    def diameterOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        if not root:
            return -1
        self.dfs(root)
        return self.max

In [ ]:
# 669
class Solution:
    def DFSTrim(self, head, low, high):
        if not head: # base case: the node is empty.
            return head
        head.left = self.DFSTrim(head.left, low, high)
        head.right = self.DFSTrim(head.right, low, high)
        if head.left:
            if head.left.val < low: # when a child's value is less than low, replace it with that child's right child;
                head.left = head.left.right
            elif head.left.val > high: # when a child's value is greater than high, replace it with that child's left child;
                head.left = head.left.left
        if head.right:
            if head.right.val > high:
                head.right = head.right.left
            elif head.right.val < low:
                head.right = head.right.right
        return head


    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        dummy_head = TreeNode(left=root) # need to create a dummy root node, to make the real root its left child
        self.DFSTrim(dummy_head, low, high)
        return dummy_head.left

In [ ]:
# 99 DFS + inorder traversal
class Solution:
    def __init__(self):
        self.left_node = None
        self.right_node = None

    def dfs(self, node, lower_bound, higher_bound):
        if not node:
            return
        # handle the left child first
        self.dfs(node.left, lower_bound, node)
        # inorder position: handle the current node between the left and right children.
        if lower_bound:
            if node.val < lower_bound.val:
                if not self.left_node:
                    self.left_node = lower_bound
                self.right_node = node
        if higher_bound:
            if node.val > higher_bound.val:
                if not self.left_node:
                    self.left_node = node
                self.right_node = higher_bound
        # finally handle the right child
        self.dfs(node.right, node, higher_bound)

    def recoverTree(self, root: Optional[TreeNode]) -> None:
        """
        Do not return anything, modify root in-place instead.
        """
        self.dfs(root, None, None)
        self.left_node.val, self.right_node.val = self.right_node.val, self.left_node.val

In [ ]:
# 1372 DFS

class Solution:
    def __init__(self):
        self.max = 0
    def dfs(self, node):
        if not node:
            return -1, -1
        _, right_path = self.dfs(node.left) # children first, then the current node.
        left_path, _  = self.dfs(node.right)
        self.max = max(self.max, right_path + 1, left_path + 1) # update the global variable
        return right_path + 1, left_path + 1

    def longestZigZag(self, root: TreeNode) -> int:
        self.dfs(root)
        return self.max


class Solution:
    def __init__(self):
        self.max = 0
    def dfs(self, node):
        right_path = -1
        if node.left: # add a check here as an early stop, to save some time.
            _, right_path = self.dfs(node.left)
        left_path = -1
        if node.right:
            left_path, _  = self.dfs(node.right)
        self.max = max(self.max, right_path + 1, left_path + 1)
        return right_path + 1, left_path + 1

    def longestZigZag(self, root: TreeNode) -> int:
        self.dfs(root)
        return self.max

In [ ]:
# 1367 Approach 1: try two kinds of DFS expansion at each node

class Solution:
    def isSubPath(self, head: ListNode, root: TreeNode) -> bool:
        print(head, root)
        if not head or (root.val == head.val and not head.next):
            return True
        if root.left:
            if root.val == head.val:
                if self.isSubPath(head.next, root.left):
                    return True
            if self.isSubPath(head, root.left):
                return True
        if root.right:
            if root.val == head.val:
                if self.isSubPath(head.next, root.right):
                    return True
            if self.isSubPath(head, root.right):
                return True
        return False

# Approach 2:
class Solution:
    def dfs(self, cur_head, root):
        # check whether the subtree starting at the current root is currently matching the list starting from cur_head
        print("dfs: ", cur_head, root)
        if not cur_head:
            return True
        if not root:
            return False
        if root.val == cur_head.val:
            return self.dfs(cur_head.next, root.left) or self.dfs(cur_head.next, root.right)
        # the current root's value differs from the current head's value, so we can prune early.
        return False

    def isSubPath(self, head: ListNode, root: TreeNode) -> bool:
        # DFS starting fresh from the original head.
        print("isSubPath: ", head, root)
        if not head:
            return True
        if not root:
            return False
        return self.dfs(head, root) or self.isSubPath(head, root.left) or self.isSubPath(head, root.right)
        


# 1367 comparison of the two approaches' complexity
# example
[1,4,2,6]
[1,4,4,null,2,2,null,1,null,6,8,null,null,null,null,1,3]


# approach one: took 13 attempts
ListNode{val: 1, next: ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 4, left: None, right: TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}}, right: TreeNode{val: 4, left: TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}, right: None}}
ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}} TreeNode{val: 4, left: None, right: TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}}
ListNode{val: 2, next: ListNode{val: 6, next: None}} TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 6, next: None} TreeNode{val: 1, left: None, right: None}
ListNode{val: 2, next: ListNode{val: 6, next: None}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}} TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}}} TreeNode{val: 4, left: None, right: TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}}
ListNode{val: 1, next: ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}}} TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 1, next: ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}} TreeNode{val: 4, left: TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}, right: None}
ListNode{val: 2, next: ListNode{val: 6, next: None}} TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}
ListNode{val: 6, next: None} TreeNode{val: 6, left: None, right: None}





# approach two: took 10 attempts
isSubPath:  ListNode{val: 1, next: ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 4, left: None, right: TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}}, right: TreeNode{val: 4, left: TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}, right: None}}
dfs:  ListNode{val: 1, next: ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 4, left: None, right: TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}}, right: TreeNode{val: 4, left: TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}, right: None}}
dfs:  ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}} TreeNode{val: 4, left: None, right: TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}}
dfs:  ListNode{val: 2, next: ListNode{val: 6, next: None}} None
dfs:  ListNode{val: 2, next: ListNode{val: 6, next: None}} TreeNode{val: 2, left: TreeNode{val: 1, left: None, right: None}, right: None}
dfs:  ListNode{val: 6, next: None} TreeNode{val: 1, left: None, right: None}
dfs:  ListNode{val: 6, next: None} None
dfs:  ListNode{val: 4, next: ListNode{val: 2, next: ListNode{val: 6, next: None}}} TreeNode{val: 4, left: TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}, right: None}
dfs:  ListNode{val: 2, next: ListNode{val: 6, next: None}} TreeNode{val: 2, left: TreeNode{val: 6, left: None, right: None}, right: TreeNode{val: 8, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 3, left: None, right: None}}}
dfs:  ListNode{val: 6, next: None} TreeNode{val: 6, left: None, right: None}
dfs:  None None
  

# example2
[1,1,1,1,1]
[1,1,1,null,1,1,null,1,null,1,1,null,null,null,null,1,1]

# approach one: took 21 attempts
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}, right: None}}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: None, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: None}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}}} TreeNode{val: 1, left: None, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}
ListNode{val: 1, next: ListNode{val: 1, next: None}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: None, right: None}
ListNode{val: 1, next: ListNode{val: 1, next: None}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}
ListNode{val: 1, next: None} TreeNode{val: 1, left: None, right: None}


    
# approach two dfs: took 17 attempts
isSubPath:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}, right: None}}
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}, right: None}}
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: None, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}}
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} None
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: None}
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: None}} TreeNode{val: 1, left: None, right: None}
dfs:  ListNode{val: 1, next: None} None
dfs:  ListNode{val: 1, next: None} None
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: None}} None
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}}} TreeNode{val: 1, left: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}, right: None}
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: ListNode{val: 1, next: None}}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}}
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: None}} TreeNode{val: 1, left: None, right: None}
dfs:  ListNode{val: 1, next: None} None
dfs:  ListNode{val: 1, next: None} None
dfs:  ListNode{val: 1, next: ListNode{val: 1, next: None}} TreeNode{val: 1, left: TreeNode{val: 1, left: None, right: None}, right: TreeNode{val: 1, left: None, right: None}}
dfs:  ListNode{val: 1, next: None} TreeNode{val: 1, left: None, right: None}
dfs:  None None



In [ ]:
# 297 Approach 1: preorder traversal, DFS + bracket matching
# Runtime: 220 ms, beats 52% of Python3 submissions
# Memory usage: 24.5 MB, beats ...% of Python3 submissions
class Codec:

    def serialize(self, root):
        """Encodes a tree to a single string.

        :type root: TreeNode
        :rtype: str
        """
        if not root:
            return ''
        res = str(root.val)
        res += ('L(' + self.serialize(root.left) + ')') # for bracket matching during deserialization
        res += ('R(' + self.serialize(root.right) + ')')
        # print(root, res)
        return res


    def deserialize(self, data):
        """Decodes your encoded data to tree.

        :type data: str
        :rtype: TreeNode
        """
        if not data:
            return None
        # print(data)
        cur_val_id = 1
        while data[cur_val_id] != 'L':
            cur_val_id += 1
        cur_val = data[:cur_val_id]
        root = TreeNode(int(cur_val))
        cur_cnt = 1
        left_end_id = cur_val_id + 2
        while cur_cnt > 0:
            if data[left_end_id] == '(':
                cur_cnt += 1
            elif data[left_end_id] == ')':
                cur_cnt -= 1
            left_end_id += 1
        root.left = self.deserialize(data[cur_val_id + 2: left_end_id-1])
        root.right = self.deserialize(data[left_end_id + 2: -1])
        return root

# A variant of bracket matching: left/right markers
# Runtime: 160 ms, beats 55% of Python3 submissions
# Memory usage: 22.3 MB, beats 6% of Python3 submissions
class Codec:
    def serialize(self, root):
        """Encodes a tree to a single string.

        :type root: TreeNode
        :rtype: str
        """
        if not root:
            return ''
        res = str(root.val)
        res += ('L' + self.serialize(root.left)) # realized L/R markers can replace the brackets, saving memory
        res += ('R' + self.serialize(root.right))
        return res


    def deserialize(self, data):
        """Decodes your encoded data to tree.

        :type data: str
        :rtype: TreeNode
        """
        if not data:
            return None
        # print(data)
        cur_val_id = 1
        while data[cur_val_id] != 'L':
            cur_val_id += 1
        cur_val = data[:cur_val_id]
        root = TreeNode(int(cur_val))

        cur_cnt = 1 # bracket-matching idea, matching the count of L and R markers.
        left_end_id = cur_val_id
        while cur_cnt > 0:
            left_end_id += 1
            if data[left_end_id] == 'L':
                cur_cnt += 1
            elif data[left_end_id] == 'R':
                cur_cnt -= 1
        root.left = self.deserialize(data[cur_val_id + 1: left_end_id])
        root.right = self.deserialize(data[left_end_id + 1: ])
        return root

# Approach 2: use a stack to deserialize
# Runtime: 108 ms, beats 92.18% of Python3 submissions
# Memory usage: 20.4 MB, beats 88.63% of Python3 submissions
class Codec:

    def serialize(self, root):
        """Encodes a tree to a single string.

        :type root: TreeNode
        :rtype: str
        """
        if not root:
            return ','
        res = (str(root.val) + ',')
        res += self.serialize(root.left)
        res += self.serialize(root.right)
        return res


    def deserialize(self, data):
        """Decodes your encoded data to tree.

        :type data: str
        :rtype: TreeNode
        """
        vals = data.split(',')[:-1]
        if len(vals) < 3:
            return []
        root = TreeNode(int(vals[0]))
        stack = [[root, 0]]
        for val in vals[1: ]:
            if val == '':
                cur_node = None
                while stack:
                    former_node, status = stack.pop()
                    if status == 1:
                        former_node.right = cur_node
                        cur_node = former_node
                    else:
                        former_node.left = cur_node
                        stack.append([former_node, 1])
                        break
            else:
                cur_node = TreeNode(int(val))
                stack.append([cur_node, 0])
        return root

In [ ]:
# 332 Approach 1: build the tree with DFS
class Solution:
    def __init__(self):
        self.itineary = None
        self.num_tickets =  0
        self.Dep2ArrCnt = {} # which airports can still be flown to
        self.usedCnt = {} # which airports have already been flown to

    def count(self, tickets):
        self.num_tickets = len(tickets)
        for depart, arriv in tickets:
            if depart not in self.Dep2ArrCnt:
                self.Dep2ArrCnt[depart] = {arriv : 1}
                self.usedCnt[depart] = {}
            else:
                self.Dep2ArrCnt[depart][arriv] = self.Dep2ArrCnt[depart].get(arriv, 0) + 1
            self.usedCnt[depart][arriv] = 0

    def DFSBuild(self, cur_itinerary):
        if len(cur_itinerary) == self.num_tickets + 1: # finished the whole itinerary, return
            self.itineary = cur_itinerary
            return
        cur_departure = cur_itinerary[-1]
        if cur_departure not in self.Dep2ArrCnt:
            return
        all_arrives = self.Dep2ArrCnt[cur_departure]
        possible = [] # all possible next-stop airports
        for arrive, cnt in all_arrives.items():
            if cur_departure in self.usedCnt and arrive in self.usedCnt[cur_departure] and self.usedCnt[cur_departure][arrive] < cnt:
                possible.append(arrive)
        for cur_arrive in sorted(possible): # sort: greedily pick the alphabetically smallest airport connected to the current node
            if self.itineary is not None: # early stop
                return
            self.usedCnt[cur_departure][cur_arrive] += 1
            self.DFSBuild(cur_itinerary + [cur_arrive])
            self.usedCnt[cur_departure][cur_arrive] -= 1

    def findItinerary(self, tickets: List[List[str]]) -> List[str]:
        self.count(tickets)
        self.DFSBuild(['JFK'])
        return self.itineary

In [ ]:
# 987
import functools
class Solution:
    def __init__(self):
        self.colrowval = []

    def BFS(self, head, cur_row, cur_col):
        self.colrowval.append([cur_col, cur_row, head.val])
        if head.left:
            self.BFS(head.left, cur_row + 1, cur_col - 1)
        if head.right:
            self.BFS(head.right, cur_row + 1, cur_col + 1)

    def verticalTraversal(self, root: TreeNode) -> List[List[int]]:
        self.BFS(root, 0, 0) # save col, row, and val, for sorting
        res = [[]]
        def cmp(x, y):
            if x[0] < y[0]:
                return -1
            elif x[0] > y[0]:
                return 1
            else:
                if x[1] < y[1]:
                    return -1
                elif x[1] > y[1]:
                    return 1
                else:
                    if x[2] < y[2]:
                        return -1
                    elif x[2] > y[2]:
                        return 1
            return 0

        self.colrowval.sort(key=functools.cmp_to_key(cmp))
        last_col = self.colrowval[0][0]
        for item in self.colrowval:
            if item[0] != last_col:
                last_col = item[0]
                res.append([])
            res[-1].append(item[2])
        return res

In [ ]:
# Interview Question 0406
class Solution:
    def __init__(self):
        self.sofar_bigger = None # use a global variable to keep the smallest node found so far that's greater than the target value.

    def inorderSuccessor(self, root: TreeNode, p: TreeNode) -> TreeNode:
        if not root:
            return self.sofar_bigger
        if root.val <= p.val: # current value <= target value, search the right subtree
            if not root.right: # early stop, not necessary
                return self.sofar_bigger
            return self.inorderSuccessor(root.right, p)

        if not self.sofar_bigger or self.sofar_bigger.val > root.val: # update the global variable based on how it compares to the current value
            self.sofar_bigger = root
        # current value > target value, search the left subtree
        if not root.left: # early stop, not necessary
            return self.sofar_bigger
        return self.inorderSuccessor(root.left, p)

# Simplified version: actually no global variable is needed, because a BST has the property that every node in the left subtree is smaller than the root — so in the solution above, if root.val is greater than p, self.sofar_bigger will always be some ancestor of the current root, whose value is guaranteed to be greater than root.val
class Solution:
    def inorderSuccessor(self, root: TreeNode, p: TreeNode) -> TreeNode:
        if not root:
            return None
        if root.val <= p.val:
            return self.inorderSuccessor(root.right, p)
        left_ans = self.inorderSuccessor(root.left, p)
        return left_ans if left_ans else root

In [ ]:
# 652
class Solution:
    def traverse(self, head):
        if head is None:
            return "#"
        left_code = self.traverse(head.left)
        right_code = self.traverse(head.right)

        cur_code = str(head.val) + "," + left_code + "," + right_code # serialize at the postorder position
        # print("Traverse for {} and get {}".format(head, cur_code))
        self.nodecnt[cur_code] = self.nodecnt.get(cur_code, 0) + 1
        # print("Nodecnt: ", self.nodecnt)
        if self.nodecnt[cur_code] == 2:
            self.ans.append(head)
        return cur_code

    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        self.ans = []
        self.nodecnt = {}
        self.traverse(root)
        return self.ans


## Focused Practice on Iterative Methods


- [(medium)105Construct Binary Tree From Preorder And Inorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-inorder-traversal/)
    - [X] 🌟💡Approach 3: iterative method: [reference solution](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-inorder-traversal/solution/cong-qian-xu-yu-zhong-xu-bian-li-xu-lie-gou-zao-9/)
    The overall idea is: iterate over the elements in preorder,
        1. For the current element, decide whether it's the left child or the right child of the top-of-stack element. How do we tell? If we only had preorder, we obviously couldn't tell whether it's a left or right child. But with inorder available too: if it's a right child, the corresponding position in inorder must be the value of its parent; otherwise, it's a left child.
        2. If it's a left child, what do we do? Set the top-of-stack element's left to the current node, and push the current node.
        3. If it's a right child, what do we do? Walk forward through inorder, repeatedly popping the stack, until we find its parent; set the current node as that parent's right child. Pop the parent, and push the current node.
        The stack holds the nodes that are still waiting to have their right child found.

- [(medium)106Construct Binary Tree From Inorder And Postorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-inorder-and-postorder-traversal/)
    - [X] 💡Approach 2: iterative method, similar to the iterative method for #105.
    Same question again: if we only had postorder, **read from back to front**, how would we know whether the second node is to the left or right of the first node? Obviously we can't — we need inorder's help.
        - If the second node is to the left of the first node, then the first element of inorder must equal the first element of postorder; otherwise, if they're not equal, the second node is to the right of the first node. The rest of the reasoning follows #105.
- [(medium)889Construct Binary Tree From Preorder And Postorder Traversal](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-postorder-traversal/).
    - [X] 💡Approach 2: iterative method, [see the reference solution's comments](https://leetcode.cn/problems/construct-binary-tree-from-preorder-and-postorder-traversal/solution/gen-ju-qian-xu-he-hou-xu-bian-li-gou-zao-er-cha-sh/)
        (Note: reconstructing from preorder + postorder alone, the answer is not unique)
        The stack holds nodes that haven't been fully visited yet: meaning, if a right subtree exists, it hasn't been visited yet; if it doesn't exist, the node itself hasn't been fully visited yet.
        How do we know when a node has been fully visited? Postorder traversal! Reading through the postorder result from front to back, whichever node we're currently reading has just become fully visited.
        The overall idea: iterate over each element in preorder, adding it to the stack. Then check whether the top-of-stack element equals the current position in postorder — if so, pop it, and set the popped element as the left or right child of the new top of stack. Loop until they're no longer equal. As for whether to set it as left or right: if left is empty, set left; otherwise set right.



Comparing these three problems:
- [X] When do we push? Following traversal order, a new node always gets pushed
- [X] When do we pop?
    - 105: pop when the top-of-stack element equals the current inorder element
    - 106: pop when the top-of-stack element equals the current inorder element
    - 889: pop when the top-of-stack element equals the current postorder element
- [X] When do we set the left subtree? When do we set the right subtree?
    - 105:
        - Left subtree: if the new node is a left child, set it as the top-of-stack's left before pushing;
        - Right subtree: if the new node is a right child, first pop repeatedly to find its parent, then set it as the old top-of-stack's right;
    - 106: symmetric to 105 (left/right swapped)
    - 889: set it during the popping process. The element to be set is the previous top-of-stack, and based on whether the new top-of-stack's left has already been set, assign it as left or right.



In [ ]:
# 105 Time 120ms->40ms, beat 97.09%; Space: 88MB->15.6MB, beat 97.71%
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> TreeNode:
        root = TreeNode(preorder[0])
        stack = [root]
        inorder_id = 0
        for i in range(1, len(preorder)):
            cur_node = TreeNode(preorder[i])
            # for the current element, decide whether it's the left or right child of the top-of-stack element.
            # if it's a left child, push it; if it's a right child, pop the top-of-stack element to find its parent, set it as that node's right child, and push the new node
            # how to tell: if the previous element equals the current head value of inorder, it's a right child; otherwise it's a left child
            if stack[-1].val == inorder[inorder_id]:
                # it's a right child
                while stack and stack[-1].val == inorder[inorder_id]:
                    father = stack.pop()
                    inorder_id += 1
                father.right = cur_node
                stack.append(cur_node)
            else:
                # it's a left child
                stack[-1].left = cur_node
                stack.append(cur_node)
        return root

In [ ]:
# 106 Time 128ms->48ms, beat 86.22%; Space: 88MB->16MB, beat 98.44%
class Solution:
    def buildTree(self, inorder: List[int], postorder: List[int]) -> TreeNode:
        root = TreeNode(postorder[-1])
        stack = [root]
        in_id = len(inorder) - 1
        for i in range(len(postorder)-2, -1, -1):
            # decide whether the current node is the left or right child of the top-of-stack node. The stack holds nodes still waiting to have their left child found
            # if it's a right child, push it;
            # if it's a left child, find its parent, set it as the parent's left child; pop the parent, and push this node
            # how to tell: if the top-of-stack element equals the value currently pointed to by inorder, it's a left child; otherwise it's a right child
            cur_node = TreeNode(postorder[i])
            if stack[-1].val != inorder[in_id]:
                stack[-1].right = cur_node
                stack.append(cur_node)
            else:
                while stack and stack[-1].val == inorder[in_id]:
                    father = stack.pop()
                    in_id -= 1
                father.left = cur_node
                stack.append(cur_node)
        return root

In [ ]:
# 889
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> TreeNode:
        root = TreeNode(preorder[0])
        post_id = 0
        stack = [root] # in the stack, save the nodes have not been completely visited.
        for i in range(1, len(preorder)):
            cur_node = TreeNode(preorder[i])
            stack.append(cur_node)
            while stack and stack[-1].val == postorder[post_id]:
                to_add_node = stack.pop()
                post_id += 1
                if stack:
                    if stack[-1].left is None:
                        stack[-1].left = to_add_node
                    else:
                        stack[-1].right = to_add_node
        return root

# My Summary

Key points for BFS problems:
1. Variation: switch from per-node granularity to per-level granularity
2. Variation: the root changes from a single node to a list of nodes
3. Pay attention to when state variables get updated — updating at the right time can cut down on redundant visits.
    - Sometimes, update the state variable before looking for children — e.g. `minutes` in #994, or `dist` in #1162. Variables like these are usually part of what the problem is asking for, and the initial value has to match where the update happens.
    - Sometimes, update the state variable while looking for children — e.g. `grid[next_i][next_j]` in #1162. Variables like this are usually used to track visited status.
    - Sometimes, update only after all children have been found — e.g. `self.begin_visited` and `self.end_visited` in the bidirectional BFS solution for #126. This is because we need to record every path, and duplicates are allowed.


Key points for DFS problems:
1. What's the base case for the recursion?
2. What operation happens at each step, and what value gets returned?
3. Usually need a global variable (often the final result), updated during the recursion.
4. What's the order relative to the left/right children? Do we operate on the current node first, then traverse left/right? Or traverse left/right first, then operate on the current node? Left first or right first? (i.e. the traversal order — preorder? inorder? postorder?) See #99 as a reference: left first, then the node itself, then right.


Using a stack during traversal: see #297 as a reference
- Be clear about what operation to perform on each element during the traversal
- Be clear about what the elements on the stack are waiting for. When do we push? When do we pop?
